In [ ]:
!pip install llamafactory
!pip install bitsandbytes -q


In [19]:
!pip install wandb -q
import wandb


In [29]:
import os
from google.colab import userdata
import wandb

# Load the key from Colab secrets
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

# Log in (this should now succeed)
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: manarabdelgawad531 (manarabdelgawad531-university-of-sadat-city) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import json ,os
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 28634, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 28634 (delta 83), reused 47 (delta 47), pack-reused 28482 (from 3)
Receiving objects: 100% (28634/28634), 14.08 MiB | 17.87 MiB/s, done.
Resolving deltas: 100% (20367/20367), done.
/content/LLaMA-Factory


In [30]:
with open("/content/LLaMA-Factory/data/egypt_history.json","r",encoding="utf-8") as f:
  data_english=json.load(f)

In [31]:
len(data_english)

203

In [37]:
with open("/content/LLaMA-Factory/data/dataset_info.json","r",encoding="utf-8") as f:
  dataset_info=json.load(f)

dataset_info["egypt_history"]={
    "file_name":"egypt_history.json",
    "formatting":"sharegpt",
    "columns":{"messages":"messages"},
    "tags":{
        "role_tag":"role",
        "content_tag":"content",
        "user_tag":"user",
        "assistant_tag":"assistant"
    }
}

In [38]:
with open("/content/LLaMA-Factory/data/dataset_info.json","w",encoding="utf-8") as f:
  json.dump(dataset_info,f,ensure_ascii=False,indent=4)

In [39]:
yaml_content="""model_name_or_path: microsoft/Phi-3.5-mini-instruct
trust_remote_code: true

stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all
quantization_bit: 4
lora_dropout: 0.05

dataset: egypt_history
template: phi
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 4

output_dir: saves/phi-egypt-qlora
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
gradient_checkpointing: true

report_to: wandb
run_name : egypt-history
"""

with open("/content/LLaMA-Factory/train_egypt_qlora.yaml","w",encoding="utf-8") as f:
  f.write(yaml_content)

In [35]:
!tail -3 /content/LLaMA-Factory/train_egypt_qlora.yaml | cat -A

$
report_to: wandb$
run_name : egypt-history$


In [40]:
%cd /content/LLaMA-Factory
!llamafactory-cli train train_egypt_qlora.yaml

/content/LLaMA-Factory
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-11-10 08:53:29] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-11-10 08:53:29] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:771] 2026-11-10 08:53:29,736 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--microsoft--Phi-3.5-mini-instruct/snapshots/2fe192450127e6a83f7441aef6e3ca586c338b77/config.json
[INFO|con

In [41]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path="/content/LLaMA-Factory/saves/phi-egypt-qlora", repo_id="Manar-Abdelgawad/phi-egypt-history-qlora", repo_type="model")


CommitInfo(commit_url='https://huggingface.co/Manar-Abdelgawad/phi-egypt-history-qlora/commit/a191f506852ad2a9a26a1bdd8104e39f2238e590', commit_message='Upload folder using huggingface_hub', commit_description='', oid='a191f506852ad2a9a26a1bdd8104e39f2238e590', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Manar-Abdelgawad/phi-egypt-history-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Manar-Abdelgawad/phi-egypt-history-qlora'), pr_revision=None, pr_num=None)